# Qwen3.5-0.8B LoRA Fine-Tuning — Colab (T4 x1)

Fine-tune Qwen3.5-0.8B on any of the 30 datasets published under
`michaelowusuntim6` on Hugging Face.

**Hardware (Colab free tier):** T4 x1, 16 GB VRAM, bf16 supported.
bf16 LoRA uses ~3 GB VRAM for Qwen3.5-0.8B, so a single T4 is plenty.
**Estimated 1 epoch on 10k records: ~15–25 minutes.**

**Session limits:** Colab free tier disconnects after ~90 minutes idle and
caps sessions at ~12 hours. Keep the tab active, or mount Google Drive and
checkpoint there so a disconnect does not lose the adapter.

Use bf16 LoRA. Do **not** use QLoRA 4-bit — Qwen3.5 is not recommended for it
and there is no VRAM pressure on a 16 GB T4 at this model size.


In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps transformers trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

DATASET_NAME = "code-review-qwen35"  # change to any of the 30 on HF
MAX_SEQ_LEN = 8192
EXPERT_NAME = "code_review"
OUT_DIR = "/content"

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-0.8B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto -> bf16 on a T4
    load_in_4bit=False,    # bf16 LoRA: QLoRA is not recommended for Qwen3.5
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
ds = load_dataset(f"michaelowusuntim6/{DATASET_NAME}", split="train")

# Pre-format with the model's native chat template. The template adds
# <|im_start|>/<|im_end|>; never insert them yourself.
def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(format_record, num_proc=2)
print(len(ds), "records")


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),   # True on T4
        fp16=False,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"{OUT_DIR}/outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("adapter saved to", f"{OUT_DIR}/adapters/{EXPERT_NAME}")

# Optional: mount Drive so an idle disconnect does not lose the adapter
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {OUT_DIR}/adapters/{EXPERT_NAME} /content/drive/MyDrive/


In [ ]:
# from huggingface_hub import login
# import os
# login(token=os.environ["HF_TOKEN"])
# model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")
# tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}-adapter")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "How do I fix a memory leak in C?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
